# 02 Tomographic geometry, wavenumbers, and steering vectors

This notebook confirms the tomographic forward operator that the coherence and covariance terms are built on. `TomoGeometry` converts physical baselines into vertical wavenumbers `kz`, forms the steering matrix `exp(j kz x)`, and the per-height outer products used to assemble covariance matrices.

Modules exercised: `tools.sar.tomo_geometry.TomoGeometry`, `configuration.training_config.GeometryConfig`.

We verify the baseline-to-kz scaling, inspect the steering phase ramps, and confirm the outer-product tensor reproduces a steering-vector outer product at a single height.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
import math

from configuration.training_config import GeometryConfig
from tools.sar.tomo_geometry            import TomoGeometry


## Build the geometry from physical baselines

The default configuration uses a 9-track baseline ladder, a wavelength of 0.23 m, and a slant range of 5000 m. The conversion is `kz = 4 pi b / (lambda r0)`.

In [ ]:
x_axis = torch.linspace(-20.0, 40.0, 160)
cfg    = GeometryConfig()
geom   = TomoGeometry(cfg, x_axis)

scale     = 4.0 * math.pi / (cfg.wavelength * cfg.slant_range)
kz_manual = scale * np.asarray(cfg.baselines, dtype=np.float64)

print("tracks            :", geom.n_tracks)
print("kz from geometry  :", np.round(geom.kz.cpu().numpy(), 4))
print("kz manual formula :", np.round(kz_manual, 4))
print("max abs difference:", float(np.abs(geom.kz.cpu().numpy() - kz_manual).max()))

## Steering phase ramps

Each track contributes a complex exponential whose phase winds linearly with elevation at a rate set by `kz`. The zero-baseline track is flat; higher baselines wind faster.

In [ ]:
phase = torch.angle(geom.steering).cpu().numpy()   # (n_tracks, n_bins)
xv    = x_axis.cpu().numpy()

fig, ax = plt.subplots(figsize=(6.8, 3.4))
for n in [0, 2, 4, 8]:
    ax.plot(xv, phase[n], lw=1.1, label=f"track {n}, kz={float(geom.kz[n]):.3f}")
ax.set_xlabel("elevation [m]")
ax.set_ylabel("steering phase [rad]")
ax.set_title("Steering-vector phase ramps per baseline")
ax.legend()
plt.show()

## Outer-product tensor

`geom.outer` has shape `(n_tracks, n_tracks, n_bins)`: for each height it is the rank-one outer product `a(x) a(x)^H` of the steering vector. Integrating it against a reflectivity profile yields the interferometric covariance. We confirm a single height slice equals the explicit outer product.

In [ ]:
print("steering shape:", tuple(geom.steering.shape))
print("outer shape   :", tuple(geom.outer.shape))

k        = 80
a_k      = geom.steering[:, k]
explicit = torch.outer(a_k, a_k.conj())
stored   = geom.outer[:, :, k]
print("max abs difference at height index 80:", float((explicit - stored).abs().max()))

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.2))
for ax, mat, title in zip(axes, [explicit.real, explicit.imag], ["Re a a^H", "Im a a^H"]):
    im = ax.imshow(mat.cpu().numpy(), cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("track")
    ax.set_ylabel("track")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Single-height steering outer product")
fig.tight_layout()
plt.show()

## Expected outcome

The geometry's `kz` reproduces the analytic baseline formula to machine precision. Phase ramps wind faster for longer baselines and are flat for the zero baseline. The stored outer-product slice equals the explicit Hermitian rank-one matrix, with a real diagonal of ones, confirming the forward operator the covariance terms integrate against.